# Wildfire Exploratory Data Analysis

## Objectives

* Explore wildfire trends across Europe, 1980-2024
* Focus analysis on Spain, Portugal, France, and Greece
* Validate project hypothesis around Southern European wildfire trends
* Produce visualisations for the dashboard (min. 2 plot types)

## Inputs

* inputs/processed/wildfires_long_format.csv

## Outputs

* Charts/insights to be reused in the Streamlit dashboard

## Additional Comments

* Focus countries chosen because they have complete EFFIS reporting since 1980
  and are central to the 2025-2026 wildfire crisis motivating this project

---

In [1]:
import pandas as pd

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [2]:
import os
current_dir = os.getcwd()
current_dir

'/Users/tildeholmqvist/Documents/VS_Code_Tilde/DA_project_3/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chdir() defines the new current directory

In [3]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [4]:
current_dir = os.getcwd()
current_dir

'/Users/tildeholmqvist/Documents/VS_Code_Tilde/DA_project_3'

# Section 1: Load Processed Data

Load the cleaned long-format dataset produced in `01_data_cleaning.ipynb`.

In [5]:
df = pd.read_csv("inputs/processed/wildfires_long_format.csv")

print(df.shape)
df.head()

(922, 5)


,Year,country_iso3,burnt_area_ha,number_of_fires,country_name
0,1980,PRT,44251.0,2349.0,Portugal
1,1981,PRT,89798.0,6730.0,Portugal
2,1982,PRT,39556.0,3626.0,Portugal
3,1983,PRT,47811.0,4539.0,Portugal
4,1984,PRT,52710.0,7356.0,Portugal


---

## Statistical Concepts

Before looking at summary statistics, here are the key terms used in this notebook:

* **Mean** — the average value. Sensitive to extreme values (outliers).
* **Std (Standard Deviation)** — measures how spread out the values are around
  the mean. A high std means values vary a lot from year to year.
* **Min / Max** — the smallest and largest recorded values.
* **25% / 50% / 75% (Quartiles)** — the value below which 25%, 50% (the median),
  or 75% of the data falls. The median is often a more reliable "typical value"
  than the mean when a dataset has extreme outliers, like an unusually severe
  wildfire year.

---

# Section 2: Focus Countries Overview

Filter the dataset to Spain, Portugal, France, and Greece, and visualise burnt
area trends over time for each.

In [6]:
focus_countries = ['Spain', 'Portugal', 'France', 'Greece']
df_focus = df[df['country_name'].isin(focus_countries)]

print(df_focus.shape)
df_focus.head()

(180, 5)


,Year,country_iso3,burnt_area_ha,number_of_fires,country_name
0,1980,PRT,44251.0,2349.0,Portugal
1,1981,PRT,89798.0,6730.0,Portugal
2,1982,PRT,39556.0,3626.0,Portugal
3,1983,PRT,47811.0,4539.0,Portugal
4,1984,PRT,52710.0,7356.0,Portugal


## Burnt Area Trends Over Time

In [7]:
import plotly.express as px

fig = px.line(df_focus, x='Year', y='burnt_area_ha', color='country_name',
              title='Burnt Area Over Time: Spain, Portugal, France, Greece (1980-2024)',
              labels={'burnt_area_ha': 'Burnt Area (hectares)', 'Year': 'Year', 'country_name': 'Country'})
fig.show()

**In Plain Language:** Each line shows one country's wildfire damage per year, measured
in hectares burnt. Spikes upward mean a particularly bad fire season for that country.
Hover over any point to see the exact year and country.

**Observation:** Spain shows the largest year-to-year swings in the 1980s and 1990s,
with several years exceeding 400,000 hectares burnt, followed by much calmer years.
Portugal recorded the single highest spike in the dataset, coinciding with the
country's most severe wildfire season on record. Greece shows an isolated but
extreme spike around 2007, matching the well-documented Greek wildfire disasters
that year. France remains consistently lower than the other three countries
throughout the entire 45-year period, with no comparable extreme spikes.

 ### Verifying the Pattern
  France's consistently lower values were checked against summary statistics to
  confirm this reflects genuine differences in wildfire severity, not a data error.

In [13]:
df_focus.groupby('country_name')['burnt_area_ha'].describe()

,count,mean,std,min,25%,50%,75%,max
country_name,,,,,,,,
France,45.0,22651.800000,20265.175745,2735.0,8169.0,15906.00,24995.0,75566.0
Greece,45.0,44469.765778,45163.156438,3517.0,13393.0,28288.46,54049.0,225734.0
Portugal,45.0,113896.822222,105189.701099,19897.0,47811.0,88867.00,140953.0,539921.0
Spain,45.0,151933.958444,105077.427235,25162.0,86122.0,120094.00,188586.0,484476.0


## Number of Fires vs Burnt Area

Burnt area alone doesn't tell the whole story — a country could have many small
fires or few very large ones. Comparing number of fires to burnt area reveals
this distinction.

In [14]:
avg_fires_focus = df_focus.groupby('country_name')[['burnt_area_ha', 'number_of_fires']].mean().reset_index()
avg_fires_focus

,country_name,burnt_area_ha,number_of_fires
0,France,22651.800000,4040.733333
1,Greece,44469.765778,1371.333333
2,Portugal,113896.822222,18214.355556
3,Spain,151933.958444,13774.844444


**In Plain Language:** This table shows, for each country, two average numbers
per year: how much land burned (measured in hectares, abbreviated "ha" — one
hectare is roughly the size of a football pitch) and how many separate fires
occurred. On their own, these two numbers are hard to compare — a country with
many fires isn't necessarily the one with the most burnt land, since fires can
vary hugely in size.

In [15]:
avg_fires_focus['ha_per_fire'] = avg_fires_focus['burnt_area_ha'] / avg_fires_focus['number_of_fires']
avg_fires_focus = avg_fires_focus.sort_values('ha_per_fire', ascending=False)
avg_fires_focus

,country_name,burnt_area_ha,number_of_fires,ha_per_fire
1,Greece,44469.765778,1371.333333,32.428123
3,Spain,151933.958444,13774.844444,11.029813
2,Portugal,113896.822222,18214.355556,6.253135
0,France,22651.800000,4040.733333,5.605864


In [17]:
fig = px.bar(avg_fires_focus, x='country_name', y='ha_per_fire',
             title='Average Fire Size by Country (Hectares per Fire)',
             labels={'ha_per_fire': 'Average Hectares per Fire', 'country_name': 'Country'},
             text_auto='.1f')
fig.show()

**In Plain Language:** This table adds a third column, `ha_per_fire`, calculated
by dividing burnt area by number of fires. It answers a simple question: on
average, how big is a single fire in each country? A value of 32 means the
average fire in that country burns about 32 hectares — roughly the size of
32 football pitches.

**Observation:** Greece stands out sharply — its average fire burns roughly
32 hectares, three times larger than Spain's (11 ha) and over five times larger
than Portugal's or France's (~6 ha each). This means Greece's wildfire problem is
driven by a smaller number of exceptionally large, destructive fires, while
Portugal and France experience many more frequent but comparatively contained
fires. Spain sits in between: fewer fires than Portugal, but each one tends to
burn roughly twice as much land.

In [18]:
fig = px.scatter(avg_fires_focus, x='number_of_fires', y='ha_per_fire',
                  text='country_name', size='burnt_area_ha',
                  title='Fire Frequency vs Average Fire Size',
                  labels={'number_of_fires': 'Average Number of Fires per Year',
                          'ha_per_fire': 'Average Hectares per Fire'})
fig.update_traces(textposition='top center')
fig.show()

**In Plain Language:** This chart plots each country by two measurements at once:
how often fires happen (left to right) and how large each fire tends to be
(bottom to top). A country in the bottom-right has many small fires; a country
in the top-left has few large fires.

**Observation:** This chart confirms the "many small fires vs. few large fires"
pattern directly: Portugal and France sit toward the bottom-right (frequent,
smaller fires), while Greece sits clearly in the top-left (infrequent but very
large fires). Spain falls in between — a moderate number of fires, but each
noticeably larger than Portugal's or France's.

---

# Section 3: Europe-Wide Comparison

Compare average annual burnt area across all 31 countries in the dataset, to see
how Spain, Portugal, France, and Greece rank against the rest of Europe.

In [9]:
avg_by_country = df.groupby('country_name')['burnt_area_ha'].mean().sort_values(ascending=False).reset_index()
avg_by_country.head(10)

,country_name,burnt_area_ha
0,Spain,151933.958444
1,Portugal,113896.822222
2,Italy,101240.444444
3,Greece,44469.765778
4,Algeria,37466.166667
5,France,22651.800000
6,Turkey,14928.371429
7,Croatia,13511.515152
8,Ukraine,9633.333333
9,Bulgaria,8758.382353


In [10]:
top15 = avg_by_country.head(15)

fig = px.bar(top15, x='burnt_area_ha', y='country_name', orientation='h',
             title='Average Annual Burnt Area by Country (Top 15, 1980-2024)',
             labels={'burnt_area_ha': 'Average Burnt Area (hectares/year)', 'country_name': 'Country'})
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

**In Plain Language:** This chart ranks all 31 countries by their average yearly
wildfire damage. The longer the bar, the more hectares burnt per year on average.
Spain tops the list — even ahead of Portugal and Italy, which are often associated
more strongly with wildfire risk in public perception.

**Observation:** Spain's average annual burnt area (152k hectares) is nearly a
third higher than Portugal's (114k) and 50% higher than Italy's (101k). Notably,
France ranks only 6th overall, with a relatively low historical average
(23k hectares/year) compared to the other three focus countries.

---

# Conclusions and Next Steps

This notebook explored wildfire patterns across Europe (1980-2024), with a focus
on Spain, Portugal, France, and Greece.

**Key takeaways:**
* Spain has the highest average annual burnt area among the four focus countries,
  and ranks highest across all 31 countries in the dataset
* France has the lowest historical average burnt area of the four focus countries
* Fire size varies dramatically by country: Greece experiences fewer but far
  larger fires on average, while Portugal and France experience more frequent
  but smaller fires

**Limitation:** This dataset records burnt area and fire counts, but not fire
cause (e.g. arson vs. natural ignition). This is addressed further in the
project's ethics documentation.

**Next step:** Build the Streamlit dashboard, reusing the visualisations and
insights developed in this notebook.